# 01 — Data Quality

Load market data, enforce **point-in-time (PIT)** discipline on fundamentals, and scan engineered features for **look-ahead leakage**. These are the first two of quantcortex's seven enforced backtesting pitfalls.

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the quantcortex repo root on the path (notebooks live in research/).
for p in ("..", "."):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

RNG = np.random.default_rng(42)

def load_prices(symbols, start="2018-01-01", periods=1800):
    """Real prices via yfinance if available, else deterministic synthetic GBM."""
    try:
        from data.providers.yfinance_provider import YFinanceProvider
        px = YFinanceProvider().get_prices(list(symbols), start=start)
        if px is not None and not px.empty and px.shape[0] > 120:
            return px.dropna(how="all").ffill().dropna()
        raise RuntimeError("empty frame")
    except Exception as exc:
        print(f"[offline] yfinance unavailable ({type(exc).__name__}); using synthetic data.")
    dates = pd.bdate_range(start, periods=periods)
    mu = RNG.normal(0.0003, 0.00015, len(symbols))
    vol = RNG.uniform(0.008, 0.018, len(symbols))
    shocks = RNG.normal(mu, vol, size=(periods, len(symbols)))
    return pd.DataFrame(100 * np.exp(np.cumsum(shocks, axis=0)),
                        index=dates, columns=list(symbols))

def synth_ohlcv(close):
    """Build an OHLCV frame around a close series (synthetic intrabar range)."""
    close = close.dropna()
    hi = close * (1 + np.abs(RNG.normal(0, 0.004, len(close))))
    lo = close * (1 - np.abs(RNG.normal(0, 0.004, len(close))))
    op = close.shift(1).fillna(close.iloc[0])
    vol = RNG.integers(1_000_000, 6_000_000, len(close)).astype(float)
    return pd.DataFrame({"open": op, "high": hi, "low": lo, "close": close, "volume": vol})

print("quantcortex research environment ready.")


quantcortex research environment ready.


## Load prices
We use a small liquid basket; falls back to synthetic data offline.

In [2]:
symbols = ["AAPL", "MSFT", "AMZN", "NVDA", "JPM", "XOM"]
prices = load_prices(symbols)
returns = prices.pct_change().dropna()
print("prices:", prices.shape)
prices.tail()

prices: (2118, 6)


,AAPL,MSFT,AMZN,NVDA,JPM,XOM
date,,,,,,
2026-06-01,306.309998,460.519989,261.260010,224.098816,296.579987,149.380005
2026-06-02,315.200012,441.309998,256.519989,222.560608,300.959991,149.559998
2026-06-03,310.260010,427.339996,250.020004,214.500000,300.850006,152.529999
2026-06-04,311.230011,428.049988,253.789993,218.660004,310.890015,152.039993
2026-06-05,307.339996,416.670013,246.029999,205.100006,312.369995,149.919998


## Point-in-time enforcement
Fundamentals must be keyed on **announcement_date**, never period_end. `PITEnforcer` raises if any feature would be known before it was announced.

In [3]:
from data.processors.pit_enforcer import PITEnforcer, PITViolationError

fund = pd.DataFrame({
    "symbol": ["AAPL", "AAPL", "MSFT"],
    "period_end": pd.to_datetime(["2023-03-31", "2023-06-30", "2023-03-31"]),
    "announcement_date": pd.to_datetime(["2023-05-04", "2023-08-03", "2023-04-25"]),
    "feature_date": pd.to_datetime(["2023-05-05", "2023-08-04", "2023-04-26"]),
    "field": ["eps", "eps", "eps"],
    "value": [1.52, 1.26, 2.45],
})
clean = PITEnforcer().enforce(fund)
print("PIT enforcement passed:", len(clean), "rows clean")

PIT enforcement passed: 3 rows clean


In [4]:
# A leaking row: feature_date BEFORE the announcement -> must raise.
bad = fund.copy()
bad.loc[0, "feature_date"] = pd.Timestamp("2023-04-01")  # before the 2023-05-04 announcement
try:
    PITEnforcer().enforce(bad)
    print("ERROR: violation not caught!")
except PITViolationError as e:
    print("Correctly raised PITViolationError:", str(e)[:120])

Correctly raised PITViolationError: PIT violation for 'AAPL': announcement_date 2023-05-04 is after feature_date 2023-04-01 (would use data not yet public)


## Look-ahead leakage scan
`LookaheadDetector` flags a feature built from *future* data (`close.shift(-1)`) while clearing strictly-causal features.

In [5]:
from data.processors.lookahead_detector import LookaheadDetector

close = prices.iloc[:, 0]
feat = pd.DataFrame(index=close.index)
feat["causal_ma"] = close.rolling(10).mean().shift(1)   # only past data
feat["causal_ret"] = close.pct_change().shift(1)
feat["LEAKY_future"] = close.shift(-1)                  # tomorrow used today

report = LookaheadDetector().scan(feat, reference=close)
print(report.summary())
assert "LEAKY_future" in report.flagged_columns
assert "causal_ma" not in report.flagged_columns
print("\nFlagged:", report.flagged_columns)

Detected 2 potential leak(s):
  - [high] LEAKY_future: trailing NaNs exceed leading NaNs (1 trailing vs 0 leading NaN(s) — fingerprint of a forward shift(-k))
  - [high] LEAKY_future: best correlation with a FUTURE lag of reference (|corr|=1.000 at lag -1 (reference shifted into the future))

Flagged: ['LEAKY_future']


## Data-quality visualisation

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
(prices / prices.iloc[0]).plot(ax=ax[0], lw=1)
ax[0].set_title("Normalised prices"); ax[0].set_ylabel("growth of $1")
returns.iloc[:, 0].hist(bins=60, ax=ax[1])
ax[1].set_title(f"{symbols[0]} daily return distribution")
plt.tight_layout()
miss = prices.isna().mean().mean()
print(f"missing data fraction: {miss:.4%}")
print("data quality checks complete.")

missing data fraction: 0.0000%
data quality checks complete.
